# Lecture 15: TF-IDF and Topic Modeling

Today we'll learn how to represent documents numerically and use that
representation to discover themes in a corpus automatically.

**Key ideas**:
- **Bag of words**: represent a document by its word counts (ignore order)
- **TF-IDF**: reweight those counts so distinctive words matter more
- **Topic modeling**: find groups of words that co-occur across documents

**Packages**: `nltk`, `scikit-learn`, `matplotlib`

If you haven't installed `scikit-learn` yet:
```
conda install scikit-learn
```

In [ ]:
import nltk
from nltk.corpus import brown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import numpy as np

## Part 1: Loading the Brown Corpus by Genre

The Brown Corpus (1961) contains about 1 million words of American English,
organized into 15 genre categories: news, fiction, government documents,
romance, science fiction, and more.

We already worked with this corpus in our NLTK lectures. Today we'll treat
each genre as a single "document" and see what TF-IDF and topic modeling
can tell us about the differences between them.

In [ ]:
nltk.download("brown", quiet=True)

genre_names = brown.categories()
print(f"Number of genres: {len(genre_names)}")
print(f"Genres: {genre_names}")

In [ ]:
# Build a list of documents, one per genre
# Each "document" is the full text of all files in that genre, joined into
# a single string
documents = []
for genre in genre_names:
    words = brown.words(categories=genre)
    text = " ".join(words)
    documents.append(text)
    print(f"{genre:20s} — {len(words):>7,} words")

print(f"\nTotal documents: {len(documents)}")

## Part 2: TF-IDF

### What is TF-IDF?

As we discussed, **TF-IDF** (Term Frequency–Inverse Document Frequency)
measures how important a word is to a particular document in a collection.

- **TF** (Term Frequency): How often does this word appear in this document?
- **IDF** (Inverse Document Frequency): How rare is this word across all documents?
- **TF × IDF**: Words score high when they're frequent in one document but rare overall.

Common words like "the" appear everywhere → low IDF → low TF-IDF.  
Distinctive words like "defendant" might appear a lot in legal texts but rarely elsewhere → high TF-IDF.

### Using scikit-learn's TfidfVectorizer

`TfidfVectorizer` does everything in one step: tokenizes the text, builds
the vocabulary, computes TF-IDF scores, and returns a matrix.

The result is a **documents × words** matrix where each cell contains
the TF-IDF score for that word in that document.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
)

tfidf_matrix = vectorizer.fit_transform(documents)

print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} documents (genres)")
print(f"  → {tfidf_matrix.shape[1]} words (features)")

### Exploring TF-IDF: What words are most distinctive per genre?

Let's look at the top TF-IDF words for each genre. These are the words
that best distinguish each genre from the others.

**Before you run the next cell**: what words do you predict will be
most distinctive for `news`? For `science_fiction`? For `religion`?

In [ ]:
feature_names = vectorizer.get_feature_names_out()

for genre_index, genre in enumerate(genre_names):
    # Get the TF-IDF scores for this genre and sort by score
    scores = tfidf_matrix[genre_index].toarray().flatten()
    top_indices = scores.argsort()[-10:][::-1]
    top_words = [(feature_names[i], scores[i]) for i in top_indices]

    words_str = ", ".join(f"{word} ({score:.3f})" for word, score in top_words)
    print(f"{genre:20s} → {words_str}")
    print()

### Discussion

- Do the top words make sense for each genre?
- Are there any surprises?
- What do you notice about the kinds of words that get high TF-IDF scores?
  (Hint: you'll often see proper nouns and domain-specific terms rather
  than common English words.)

## Part 3: Topic Modeling with NMF

Now let's try to discover "topics" in the Brown Corpus automatically.

We'll use **NMF** (Non-negative Matrix Factorization), which takes our
TF-IDF matrix and factors it into two smaller matrices:

- A **topics × words** matrix (what words define each topic)
- A **documents × topics** matrix (how much each document relates to each topic)

We have to choose the number of topics in advance. Let's start with 5
topics for 15 genres.

**Before you run the next cell**: We're asking the model to find 5 topics
in 15 genres of text. What do you think a "topic" will look like?

In [ ]:
NUM_TOPICS = 5

nmf_model = NMF(
    n_components=NUM_TOPICS,
    random_state=42,
)

# Fit the model and transform documents into topic space
# document_topics has shape (num_documents, NUM_TOPICS)
document_topics = nmf_model.fit_transform(tfidf_matrix)

In [ ]:
# Display the top words for each topic
NUM_TOP_WORDS = 10

for topic_index in range(NUM_TOPICS):
    top_indices = nmf_model.components_[topic_index].argsort()[-NUM_TOP_WORDS:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f"Topic {topic_index + 1}: {', '.join(top_words)}")

### Which genres load on which topics?

Each genre gets a score for each topic. Higher scores mean that genre is
more associated with that topic. Let's see which topic each genre is most
associated with.

In [ ]:
def topic_entropy(scores):
    """Compute entropy of a topic distribution (higher = more spread out)."""
    # Normalize to a probability distribution
    total = scores.sum()
    if total == 0:
        return 0.0
    probs = scores / total
    # Avoid log(0) by filtering out zero entries
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# Sort genres by entropy (low to high)
# Low entropy = dominated by one topic; high entropy = spread across topics
genre_entropies = []
for genre_index, genre in enumerate(genre_names):
    scores = document_topics[genre_index]
    genre_entropies.append((genre, scores, topic_entropy(scores)))

genre_entropies.sort(key=lambda x: x[2])

for genre, scores, entropy in genre_entropies:
    top_topic = scores.argmax()
    scores_str = "  ".join(f"T{i+1}:{s:.2f}" for i, s in enumerate(scores))
    print(f"{genre:20s}  {scores_str}  ← mostly Topic {top_topic + 1}  (H={entropy:.2f})")

### Discussion

- Do the topics seem coherent? Can you give each topic a human-readable label?
- Do genres that you'd expect to be similar end up on the same topic?
- Some genres might load on multiple topics — what does that mean?

### Challenge: Try different numbers of topics

Go back and change `NUM_TOPICS` to 3, 8, or 15. How do the results change?
When you ask for more topics, do they become more specific? When you ask
for fewer, do they become broader?

---

## Part 4: Topic Modeling on Night Vale

Let's try the same technique on the *Welcome to Night Vale* transcript.
Night Vale is a podcast presented as community radio broadcasts from a
fictional desert town. Each episode ends (or transitions) with the phrase
**"Welcome to Night Vale"**, so we can use that as a rough episode
boundary.

Make sure you've downloaded `Night_Vale.txt` from Blackboard and placed it
in the `data/` folder of the course materials repo.

In [ ]:
import re
from pathlib import Path

night_vale_path = Path("../data/Night_Vale.txt")

if not night_vale_path.exists():
    print("Night_Vale.txt not found! Download it from Blackboard and place")
    print("it in the data/ folder of the course materials repo.")
else:
    night_vale_text = night_vale_path.read_text(encoding="utf-8")
    print(f"Loaded {len(night_vale_text):,} characters")

### Segmenting into episodes

We'll split the text on sentences that end with "Welcome to Night Vale"
(and minor variations). Each segment becomes a "document" for topic modeling.

This is a practical text processing task — exactly the kind of thing we
practiced with regex earlier in the semester.

In [ ]:
# Split on "Welcome to Night Vale" (with possible punctuation)
episodes = re.split(r"Welcome to Night Vale[.!]?", night_vale_text)

# Filter out empty or very short segments
episodes = [ep.strip() for ep in episodes if len(ep.strip()) > 200]

print(f"Number of episode segments: {len(episodes)}")
for i, ep in enumerate(episodes[:5]):
    preview = ep[:80].replace("\n", " ")
    print(f"  Episode {i+1}: {preview}...  ({len(ep):,} chars)")

In [ ]:
# Build TF-IDF matrix for Night Vale episodes
nv_vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
)
nv_tfidf = nv_vectorizer.fit_transform(episodes)
nv_feature_names = nv_vectorizer.get_feature_names_out()

print(f"TF-IDF matrix: {nv_tfidf.shape[0]} episodes × {nv_tfidf.shape[1]} words")

In [ ]:
# Fit NMF topic model
NV_NUM_TOPICS = 5

nv_nmf = NMF(n_components=NV_NUM_TOPICS, random_state=42)
nv_doc_topics = nv_nmf.fit_transform(nv_tfidf)

# Display topics
for topic_index in range(NV_NUM_TOPICS):
    top_indices = nv_nmf.components_[topic_index].argsort()[-10:][::-1]
    top_words = [nv_feature_names[i] for i in top_indices]
    print(f"Topic {topic_index + 1}: {', '.join(top_words)}")

### Which episodes load on which topics?

Just like we did with Brown Corpus genres, let's see the topic weights
for each episode.

In [ ]:
# Sort episodes by entropy (low to high)
ep_entropies = []
for ep_index in range(len(episodes)):
    scores = nv_doc_topics[ep_index]
    ep_entropies.append((ep_index, scores, topic_entropy(scores)))

ep_entropies.sort(key=lambda x: x[2])

for ep_index, scores, entropy in ep_entropies:
    top_topic = scores.argmax()
    scores_str = "  ".join(f"T{i+1}:{s:.2f}" for i, s in enumerate(scores))
    print(f"Episode {ep_index + 1:3d}  {scores_str}  ← mostly Topic {top_topic + 1}  (H={entropy:.2f})")

### Discussion

- Do the Night Vale topics pick up on recurring themes or characters?
- How do these results compare to the Brown Corpus topics?
- Night Vale has many more "documents" (episodes) than the Brown Corpus
  had (genres). Does that seem to help or hurt the topic model?

### Challenge exercises

<details>
<summary>Challenge 1: Try LDA instead of NMF</summary>

Scikit-learn also has Latent Dirichlet Allocation (LDA), another popular
topic modeling algorithm. Try swapping NMF for LDA:

```python
from sklearn.decomposition import LatentDirichletAllocation

# Note: LDA works better with raw counts than TF-IDF
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_features=3000, stop_words="english")
count_matrix = count_vectorizer.fit_transform(episodes)

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
)
lda_doc_topics = lda_model.fit_transform(count_matrix)
```

How do the LDA topics compare to the NMF topics?
</details>

<details>
<summary>Challenge 2: Visualize the document-topic distribution</summary>

Make a heatmap showing which episodes are most associated with which topics:

```python
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.imshow(nv_doc_topics.T, aspect="auto", cmap="YlOrRd")
plt.xlabel("Episode")
plt.ylabel("Topic")
plt.yticks(range(NV_NUM_TOPICS), [f"Topic {i+1}" for i in range(NV_NUM_TOPICS)])
plt.colorbar(label="Topic weight")
plt.title("Night Vale: Topic Distribution Across Episodes")
plt.tight_layout()
plt.show()
```
</details>

## Summary

| Concept | What it does | Key tool |
|---------|-------------|----------|
| Bag of words | Represents documents as word counts | `CountVectorizer` |
| TF-IDF | Reweights counts to emphasize distinctive words | `TfidfVectorizer` |
| NMF | Discovers topics (groups of co-occurring words) | `sklearn.decomposition.NMF` |
| LDA | Another topic model (probabilistic) | `LatentDirichletAllocation` |

**For your projects**: If your project involves a collection of texts
(documents, transcripts, chapters, etc.), topic modeling can help you
explore what's in the corpus before committing to a specific analysis.
TF-IDF on its own is also useful for finding the most distinctive
vocabulary in different subcorpora.

**Next time**: We'll look at *word-level* representations — word vectors,
Word2Vec, and contextual embeddings (BERT).